# 🟢 Gold Layer - Business KPIs & Analytics

## 📌 Objective

Create business-ready aggregated datasets.

## 📊 KPIs

* Player performance
* Species success rate
* Map revenue analysis
* Daily trends

## 🧠 Transformations

* Aggregations (SUM, COUNT, AVG)
* Success rate calculation
* Business metrics

## 📥 Source

* Silver Events

## 📤 Target

* Gold Tables (`game_analytics.gold.*`)

## 🎯 Goal

Provide datasets for dashboards and reporting


In [0]:
from pyspark.sql import functions as sf

In [0]:
gold_path = "abfss://curated@storageaccgameanalytics.dfs.core.windows.net/gold"
events = spark.read.table('game_analytics.silver.events')

## Player Performance
#### How good is each player?

In [0]:
player_performance = (
    events
    .groupBy('player_id', "full_name")
    .agg(
        sf.count("*").alias("total_attempts"),
        sf.sum(sf.when(
            sf.col('success_flag'), 1
        ).otherwise(0)).alias('successful_catches'),
        sf.round(
            sf.sum(sf.when(sf.col('success_flag'), 1).otherwise(0)) / sf.count('*'), 2
        ).alias('success_rate'),
        sf.avg('weight').alias('avg_weight'),
        sf.sum('price').alias('total_earnings')
    )
)
display(player_performance)

## Species Analysis
#### Which Species are hardest or most valuable?

In [0]:
species_performance = (
    events
    .groupBy('species_id', 'species_name')
    .agg(
        sf.count("*").alias("attempts"),
        sf.sum(sf.when(sf.col("success_flag"), 1).otherwise(0)).alias("successes"),
        sf.round(
            sf.sum(sf.when(sf.col("success_flag"), 1).otherwise(0)) / sf.count("*"),
            2
        ).alias("success_rate"),
        sf.avg("price").alias("avg_price")
    )
)
display(species_performance)

## Map Performance
#### Which maps generate the most value?

In [0]:
map_performance = (
    events
    .groupBy("map_id", "map_name")
    .agg(
        sf.count("*").alias("total_events"),
        sf.sum("price").alias("total_revenue"),
        sf.avg("price").alias("avg_price")
    )
)

display(map_performance)

### Daily Trends
#### How does activity change over time?

In [0]:
daily_trends = (
    spark.read.table("game_analytics.silver.events")
    .groupBy("event_date")
    .agg(
        sf.count("*").alias("total_events"),
        sf.sum("price").alias("daily_revenue"),
        sf.avg("price").alias("avg_price")
    )
)
display(daily_trends)

### High-Value Catches

In [0]:
high_value = (
    events.filter(sf.col("price") > 500)
)
display(high_value)

In [0]:
player_performance.write\
    .format('delta')\
    .mode('overwrite')\
    .option('path', f'{gold_path}/player_performance')\
    .saveAsTable('game_analytics.gold.player_performance')

species_performance.write\
    .format('delta')\
    .mode('overwrite')\
    .option('path', f'{gold_path}/species_performance')\
    .saveAsTable('game_analytics.gold.species_performance')

map_performance.write\
    .format('delta')\
    .mode('overwrite')\
    .option('path', f'{gold_path}/map_performance')\
    .saveAsTable('game_analytics.gold.map_performance')

daily_trends.write\
    .format('delta')\
    .mode('overwrite')\
    .option('path', f'{gold_path}/daily_trends')\
    .saveAsTable('game_analytics.gold.daily_trends')